# 07 - Insights Data Wrangling

Notebook ini memproses dataset `insights.csv`.

Tabel ini menyimpan insight naratif yang dapat merujuk ke sumber harian atau mingguan. Karena itu, validasi utama diarahkan pada konsistensi `period_type`, source id, dan kelengkapan teks insight.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Mengatur tampilan dataframe agar output notebook lebih mudah dibaca.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Cari root project otomatis
# Mengambil lokasi kerja notebook saat ini.
current_path = Path.cwd().resolve()

# Menelusuri parent folder sampai menemukan root project yang memiliki folder data/raw.
for path in [current_path] + list(current_path.parents):
    if (path / "data" / "raw").exists():
        PROJECT_ROOT = path
        break

# Menentukan folder sumber data raw.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
# Menentukan folder output data hasil cleaning.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
# Menentukan folder output report dan validation summary.
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

# Membuat folder processed jika belum tersedia.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# Membuat folder reports jika belum tersedia.
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Menampilkan path project untuk memastikan notebook membaca folder yang benar.
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORT_DIR   :", REPORT_DIR)

PROJECT_ROOT : C:\Data Codingan\student_stress_data_science
RAW_DIR      : C:\Data Codingan\student_stress_data_science\data\raw
PROCESSED_DIR: C:\Data Codingan\student_stress_data_science\data\processed
REPORT_DIR   : C:\Data Codingan\student_stress_data_science\outputs\reports


## 1. Load Dataset

In [ ]:
# memuat dataset dari folder yang sesuai dan menampilkan sampel awal data.
# Membaca file CSV ke dalam dataframe.
insights = pd.read_csv(RAW_DIR / "insights.csv")
users_clean = pd.read_csv(PROCESSED_DIR / "users_clean.csv")
stress_predictions_clean = pd.read_csv(PROCESSED_DIR / "stress_predictions_clean.csv")
weekly_summaries_clean = pd.read_csv(PROCESSED_DIR / "weekly_summaries_clean.csv")

# Menampilkan beberapa baris awal untuk memahami bentuk data.
insights.head()

,id,user_id,stress_prediction_id,weekly_summary_id,period_type,insight_text,created_at
0,1,1,1.0,NaN,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-02 00:01:00
1,2,1,2.0,NaN,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-02 20:46:00
2,3,1,3.0,NaN,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-03 20:52:00
3,4,1,4.0,NaN,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-05 00:59:00
4,5,1,5.0,NaN,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-05 23:12:00


## 2. Assessing Data

In [ ]:
# Menampilkan struktur kolom, tipe data, dan jumlah non-null.
insights.info()

<class 'pandas.DataFrame'>
RangeIndex: 29965 entries, 0 to 29964
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    29965 non-null  int64  
 1   user_id               29965 non-null  int64  
 2   stress_prediction_id  26365 non-null  float64
 3   weekly_summary_id     3600 non-null   float64
 4   period_type           29965 non-null  str    
 5   insight_text          29965 non-null  str    
 6   created_at            29965 non-null  str    
dtypes: float64(2), int64(2), str(3)
memory usage: 1.6 MB


In [ ]:
# menampilkan ringkasan statistik numerik dan kategorikal.
# Menampilkan ringkasan statistik untuk kolom numerik dan kategorikal.
insights.describe(include='all')

,id,user_id,stress_prediction_id,weekly_summary_id,period_type,insight_text,created_at
count,29965.000000,29965.000000,26365.000000,3600.000000,29965,29965,29965
unique,NaN,NaN,NaN,NaN,4,78,18669
top,NaN,NaN,NaN,NaN,daily,Stress score hari ini sedang. Faktor yang perl...,2026-02-25 21:03:00
freq,NaN,NaN,NaN,NaN,26286,4531,303
mean,14983.000000,150.546805,13505.331766,1800.500000,NaN,NaN,NaN
std,8650.294744,86.629058,7796.995406,1039.374812,NaN,NaN,NaN
min,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN
25%,7492.000000,76.000000,6754.000000,900.750000,NaN,NaN,NaN
50%,14983.000000,151.000000,13528.000000,1800.500000,NaN,NaN,NaN
75%,22474.000000,226.000000,20258.000000,2700.250000,NaN,NaN,NaN


In [ ]:
# menilai missing value, duplicate, dan kandidat masalah kualitas data.
print("Missing value:")
# Menghitung jumlah missing value pada setiap kolom.
print(insights.isna().sum())

# Membersihkan whitespace dan menstandarkan format teks.
period_type = insights["period_type"].astype(str).str.strip().str.lower()

daily_rows = period_type == "daily"
weekly_rows = period_type == "weekly"

daily_source_valid = (
    insights.loc[daily_rows, "stress_prediction_id"].notna()
    & insights.loc[daily_rows, "weekly_summary_id"].isna()
)

weekly_source_valid = (
    insights.loc[weekly_rows, "weekly_summary_id"].notna()
    & insights.loc[weekly_rows, "stress_prediction_id"].isna()
)

print("\nPeriod type unique:")
# Melihat variasi nilai unik untuk menilai konsistensi kategori atau format.
print(insights["period_type"].unique())

print("\nDaily source valid:", daily_source_valid.all())
print("Weekly source valid:", weekly_source_valid.all())

Missing value:
id                          0
user_id                     0
stress_prediction_id     3600
weekly_summary_id       26365
period_type                 0
insight_text                0
created_at                  0
dtype: int64

Period type unique:
<StringArray>
['daily', 'DAILY', 'weekly', 'WEEKLY']
Length: 4, dtype: str

Daily source valid: True
Weekly source valid: True


## Insight:

Struktur `insights` mirip dengan `recommendations`, yaitu mendukung sumber daily dan weekly. Oleh sebab itu, missing value pada salah satu source id harus dianalisis berdasarkan `period_type`.

Cleaning diarahkan pada standardisasi teks, validasi source id, dan memastikan setiap insight memiliki teks yang valid serta relasi ke sumber yang sesuai.

## 3. Cleaning Data

Langkah cleaning:

1. Mengubah key dan source id ke tipe numerik.
2. Menstandarkan `period_type` ke lowercase.
3. Membersihkan teks insight.
4. Menghapus baris dengan informasi umum yang tidak valid.
5. Memvalidasi `user_id`.
6. Mempertahankan hanya insight yang source id-nya sesuai dengan `period_type`.

In [ ]:
# membuat salinan dataframe lalu menjalankan proses cleaning sesuai hasil assessing.
insights_clean = insights.copy()

# Mengubah kolom ke tipe numerik; nilai yang gagal dikonversi menjadi NaN.
insights_clean["id"] = pd.to_numeric(insights_clean["id"], errors="coerce")
insights_clean["user_id"] = pd.to_numeric(insights_clean["user_id"], errors="coerce")
insights_clean["stress_prediction_id"] = pd.to_numeric(insights_clean["stress_prediction_id"], errors="coerce")
insights_clean["weekly_summary_id"] = pd.to_numeric(insights_clean["weekly_summary_id"], errors="coerce")
# Membersihkan whitespace dan menstandarkan format teks.
insights_clean["period_type"] = insights_clean["period_type"].astype(str).str.strip().str.lower()
insights_clean["insight_text"] = insights_clean["insight_text"].astype(str).str.strip()
# Mengubah kolom ke tipe datetime; format yang tidak valid menjadi NaT.
insights_clean["created_at"] = pd.to_datetime(insights_clean["created_at"], errors="coerce")

# Menghapus baris yang kehilangan kolom kunci atau informasi penting.
insights_clean = insights_clean.dropna(
    subset=["id", "user_id", "period_type", "insight_text", "created_at"]
)

insights_clean = insights_clean[
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    insights_clean["user_id"].isin(set(users_clean["id"]))
]

insights_clean = insights_clean[
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    insights_clean["period_type"].isin(["daily", "weekly"])
]

valid_prediction_ids = set(stress_predictions_clean["id"])
valid_weekly_ids = set(weekly_summaries_clean["id"])

daily_mask = insights_clean["period_type"] == "daily"
weekly_mask = insights_clean["period_type"] == "weekly"

daily_valid = (
    daily_mask
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    & insights_clean["stress_prediction_id"].isin(valid_prediction_ids)
    & insights_clean["weekly_summary_id"].isna()
)

weekly_valid = (
    weekly_mask
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    & insights_clean["weekly_summary_id"].isin(valid_weekly_ids)
    & insights_clean["stress_prediction_id"].isna()
)

insights_clean = insights_clean[daily_valid | weekly_valid]

insights_clean["id"] = insights_clean["id"].astype(int)
insights_clean["user_id"] = insights_clean["user_id"].astype(int)
insights_clean["stress_prediction_id"] = insights_clean["stress_prediction_id"].astype("Int64")
insights_clean["weekly_summary_id"] = insights_clean["weekly_summary_id"].astype("Int64")
insights_clean["created_at"] = insights_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

insights_clean = insights_clean[
    [
        "id", "user_id", "stress_prediction_id", "weekly_summary_id",
        "period_type", "insight_text", "created_at"
    ]
# Mengurutkan data agar proses deduplikasi atau output lebih stabil.
].sort_values("id")

# Menampilkan beberapa baris awal untuk memahami bentuk data.
insights_clean.head()

,id,user_id,stress_prediction_id,weekly_summary_id,period_type,insight_text,created_at
0,1,1,1,<NA>,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-02 00:01:00
1,2,1,2,<NA>,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-02 20:46:00
2,3,1,3,<NA>,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-03 20:52:00
3,4,1,4,<NA>,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-05 00:59:00
4,5,1,5,<NA>,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-05 23:12:00


## Insight Setelah Cleaning:

`insights_clean` telah mempertahankan hanya insight yang memiliki source valid sesuai jenis periodenya. Dengan demikian, data insight dapat digunakan sebagai output pendukung tanpa mengganggu analisis utama berbasis aktivitas dan prediksi.

## 4. Validation dan Save Output

In [ ]:
# membuat tabel validasi untuk memastikan hasil cleaning memenuhi aturan kualitas data.
daily_mask = insights_clean["period_type"] == "daily"
weekly_mask = insights_clean["period_type"] == "weekly"

daily_valid = (
    insights_clean.loc[daily_mask, "stress_prediction_id"].notna()
    & insights_clean.loc[daily_mask, "weekly_summary_id"].isna()
).all()

weekly_valid = (
    insights_clean.loc[weekly_mask, "weekly_summary_id"].notna()
    & insights_clean.loc[weekly_mask, "stress_prediction_id"].isna()
).all()

# Membuat dataframe validasi untuk mendokumentasikan hasil pengecekan kualitas data.
validation = pd.DataFrame([
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    {"rule": "period_type valid", "passed": insights_clean["period_type"].isin(["daily", "weekly"]).all()},
    {"rule": "daily source valid", "passed": daily_valid},
    {"rule": "weekly source valid", "passed": weekly_valid},
])

validation

,rule,passed
0,period_type valid,True
1,daily source valid,True
2,weekly source valid,True


In [ ]:
# menyimpan output hasil cleaning atau report ke folder tujuan.
# Menyimpan dataframe ke file CSV.
insights_clean.to_csv(PROCESSED_DIR / "insights_clean.csv", index=False)
validation.to_csv(REPORT_DIR / "insights_validation.csv", index=False)

print("Saved:", PROCESSED_DIR / "insights_clean.csv")

Saved: C:\Data Codingan\student_stress_data_science\data\processed\insights_clean.csv
